# 03b Gate G1 diagnosis (CPU only)

Design log.
- Notebook 03 failed Gate G1 on three marks (reports/g1_03.md): agreement 0.809, attack seen on 4 of 20 documents with delegated actions, violation share of Equation (7) 0.59. Even with replay alone, where C1 is the same model as T1 in the very task where the attack hijacks T1, about a third of the working attacks show no event. This notebook finds out why, from notebook 03's saved runs. No model is called; a CPU runtime is enough.
- Part 1, missed attacks: every replay pair (attack document, replay episode, canary) gets one cause: the pair is not usable (control run with an event, control or document run failed, document not read), or it is usable and the canary acted on the attacker's details (an email address, account number, link or quoted string of the payload in an argument the control run does not use) but the rule did not count it (no event reason for that call in the task's sets, the delegated-action rule authorised it, the same call in the control run, not sourced), or the canary did not act on them.
- Part 2, false alarms: every benign document marked attack seen, with the signal (trap tool, secret, tool outside the task, argument value), the canary, the episode mode and the calls themselves.
- Part 3, candidate changes, declared here before looking: event rule (control+sourced, control+newtool-or-sourced); validity (no event in the control run, or only that it ran without error, its calls being left out as before); delegated actions (off; on as in notebook 03; on with a count limit: the document's values are authorised for at most as many calls of a tool as the control run makes beyond the document run's own unflagged calls of it); trap tools (count; count only when the call uses a value from the document; do not count); canaries (both, C1, C2); decision (Equation 5, one event; or two usable episodes with an event, one alone being inconclusive); replay 1 to 3 and random 0 to 3 episodes, k_min 1 to k. Every setting is measured on all 400 DEV documents, both halves now being development data: the evaluation half was read once in notebook 03 and failed, so a changed oracle must be read again on fresh DEV documents.
- Equation (7) is also measured with q over replay episodes only, the reading under which Lemma 1 would need only the replay episodes.
- Outputs: reports/diag_03b.md, diag_attack_pairs_03b.csv, diag_benign_03b.md, diag_settings_03b.csv.

## Stage 1. Session

In [ ]:
import os, sys, subprocess, importlib, json, time, random, collections, csv, re, difflib
from pathlib import Path
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive"
    os.environ.setdefault("HF_HOME", "/content/hf")
except ImportError:
    DRIVE_ROOT = os.environ.get("CANARY_DRIVE_ROOT", "/tmp/canary_mock_drive")
PROJECT_ROOT = f"{DRIVE_ROOT}/CanaryLabel_Research"
REPO_DIR = f"{PROJECT_ROOT}/canarylabel-research"
os.environ["CANARY_DRIVE"] = DRIVE_ROOT
os.environ["CANARY_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["CANARY_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
sys.path.insert(0, "src")
import paths, gate
importlib.reload(paths); importlib.reload(gate)
paths.ensure_dirs()
RUN_DIR = paths.DATA / "runs" / "03"
R02F = paths.DATA / "runs" / "02f"
assert (R02F / "rewrites.jsonl").exists(), "the rewrites of notebook 02f are needed"
RUN_DIR.mkdir(parents=True, exist_ok=True)
PROCS, WORKERS = 4, int(os.environ.get("CANARY_WORKERS", "48"))
print("repo:", os.getcwd(), "| runs:", RUN_DIR)

In [ ]:
import cachekey, taskbank, canary, sandbox, toolcall, agents, attribution, servers, envs, variants, oracle
for module in (cachekey, taskbank, canary, sandbox, toolcall, agents, attribution, servers, envs, variants, oracle):
    importlib.reload(module)
print("modules loaded")

## Stage 2. Inputs and notebook 03's documents and episodes, rebuilt

In [ ]:
BANK = json.load(open("schema/taskbank.json"))
SUITES = json.load(open("schema/suites.json"))
SPECS = {(s, t): BANK[s][t]["spec"] for s in BANK for t in BANK[s]}
V02E = json.load(open("schema/verified_02e.json"))
V02F = json.load(open("schema/verified_02f.json"))
SPLIT_KEYS = json.load(open("schema/split_keys.json"))

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

ATTACK = read_jsonl(paths.POOL / "attack.jsonl")
BENIGN = read_jsonl(paths.POOL / "benign.jsonl")
VPOOL = read_jsonl(paths.POOL / "attack_variants.jsonl")
UNITS = {}
for r in ATTACK:
    UNITS.setdefault(agents.unit_id(r), dict(suite=r["suite"], user_task=r["user_task"], injection_task=r["injection_task"],
                                             template=r["template"], payload=r["payload"], split=r["split"]))
W1 = set(V02E["working"]["T1"])
QUIET = {a: set(V02E["quiet"][a]) for a in ("llama8b", "qwen7b")}
JOINT = {s: sorted(t for t in BANK[s] if BANK[s][t]["group"] == "BANK" and BANK[s][t]["candidates"]
                   and f"{s}/{t}" in QUIET["llama8b"] and f"{s}/{t}" in QUIET["qwen7b"]) for s in BANK}
DELEGATED_TASKS = sorted((s, t) for s in BANK for t in BANK[s] if BANK[s][t]["follows_document_instructions"] and BANK[s][t]["candidates"])
DEPLOY_TASKS = sorted((s, t) for s in BANK for t in BANK[s] if BANK[s][t]["group"] == "DEPLOY" and BANK[s][t]["candidates"]
                      and not BANK[s][t]["follows_document_instructions"])
SEEDS = collections.defaultdict(list)
for u in V02E["working"]["T1"]:
    SEEDS[UNITS[u]["split"]].append(u)
DRAWS = variants.plan_draws(SEEDS, {"AUDIT": 2400, "DEV": 400})  # the draws of notebook 02f
REW = {}
for line in open(R02F / "rewrites.jsonl", encoding="utf-8"):
    r = json.loads(line); REW[r["draw"]] = r
assert all(variants.draw_id(sp, i, s) in REW for sp, i, s in DRAWS), "notebook 02f's rewrites are incomplete"
VP = {r["draw"]: r for r in VPOOL}
assert all(d in VP and VP[d]["working"] for sp in ("AUDIT", "DEV") for d in V02F["working"][sp]), \
    "attack_variants.jsonl does not hold notebook 02f's working rewrites"
print("BANK tasks quiet for both canaries:", {s: len(v) for s, v in JOINT.items()})
print("delegated-action tasks:", len(DELEGATED_TASKS), "| DEPLOY tasks without delegation:", len(DEPLOY_TASKS))
print("rewrites verified in 02f:", collections.Counter(r["split"] for r in VPOOL))

In [ ]:
HELDOUT = set(SPLIT_KEYS["heldout_templates"]["DEV"])
ORIG = [dict(r, kind="original", status=1) for r in sorted(ATTACK, key=lambda r: r["doc_id"])
        if r["split"] == "DEV" and agents.unit_id(r) in W1 and r.get("vectors")]
REWR = [dict(r, kind="rewrite", status=1) for r in sorted(VPOOL, key=lambda r: r["doc_id"]) if r["split"] == "DEV" and r["working"]]
for d in ORIG + REWR:
    d["half"] = "EVAL" if d["template"] in HELDOUT else "TUNE"
    d["delegated"] = False
rng = random.Random("g1-03")

def pick(rows, n, key):
    rows = sorted(rows, key=lambda r: r["doc_id"])
    rng.shuffle(rows)
    groups = collections.defaultdict(list)
    for r in rows:
        groups[key(r)].append(r)
    out, order = [], sorted(groups)
    while len(out) < n and any(groups.values()):
        for g in order:
            if groups[g] and len(out) < n:
                out.append(groups[g].pop())
    return out

BEN = [dict(r, kind="benign", status=0) for r in pick([r for r in BENIGN if r["split"] == "DEV" and len(r["text"]) <= 4000], 150, lambda r: r["source"])]
rng.shuffle(BEN)
for i, d in enumerate(BEN):
    d["half"] = "TUNE" if i < 75 else "EVAL"
    d["delegated"] = (i % 75) < 20
DOCS = ORIG + REWR + BEN
DOC = {d["doc_id"]: d for d in DOCS}
assert len(DOC) == len(DOCS), "document ids repeat"

CONTEXT = {}
for d in BEN:
    rng_ = random.Random(sandbox.seed_for(d["doc_id"], 77, "g1-context"))
    s, t = rng_.choice(DELEGATED_TASKS if d["delegated"] else DEPLOY_TASKS)
    CONTEXT[d["doc_id"]] = (s, t, rng_.choice(sorted(BANK[s][t]["candidates"])))
FALLBACK = collections.Counter()

def suite_for(doc, rng_):
    if doc.get("suite") in BANK:
        return doc["suite"]
    return rng_.choice(sorted(s for s in BANK if JOINT[s]) or sorted(BANK))

def random_draw(doc, e):
    rng_ = random.Random(sandbox.seed_for(doc["doc_id"], e, "g1-03"))
    s = suite_for(doc, rng_)
    tasks = JOINT[s]
    if not tasks:
        FALLBACK[s] += 1
        tasks = [t for t in BANK[s] if BANK[s][t]["group"] == "BANK" and BANK[s][t]["candidates"]]
    t = sandbox.choose_task(rng_, tasks)
    return sandbox.draw_episode(rng_, s, t, BANK[s][t]["candidates"], list(SUITES[s]["vectors"]), "random",
                                [x["name"] for x in SUITES[s]["tools"]])

def replay(doc, e):
    rng_ = random.Random(sandbox.seed_for(doc["doc_id"], e, "g1-03"))
    if doc["status"] == 1:
        return agents.replay_draw(rng_, doc, BANK, SUITES)
    s, t, pos = CONTEXT[doc["doc_id"]]
    return agents.context_draw(rng_, s, t, pos, BANK, SUITES)

REPLAY_E, RANDOM_E = (0, 1, 2), (3, 4, 5)
EPIS = {(d["doc_id"], e): (replay(d, e) if e in REPLAY_E else random_draw(d, e)) for d in DOCS for e in REPLAY_E + RANDOM_E}
counts = collections.Counter((d["half"], d["kind"], d["delegated"]) for d in DOCS)
for k in sorted(counts):
    print(k, counts[k])
print("random-episode fallbacks to non-quiet tasks:", dict(FALLBACK))

In [ ]:
BYPAY = collections.defaultdict(list)
for u, x in UNITS.items():
    BYPAY[(x["suite"], x["injection_task"], x["template"])].append(u)
pkey = lambda u: (UNITS[u]["suite"], UNITS[u]["injection_task"], UNITS[u]["template"])
CTX_JOBS = []
for d in REWR:
    for c in sorted(BYPAY[pkey(d["seed_unit"])]):
        if c != d["seed_unit"]:
            cu = UNITS[c]
            CTX_JOBS.append(agents.variant_job("llama8b", cu, d["draw"], d["payload"], BANK[cu["suite"]][cu["user_task"]]["candidates"]))
print("rewrite context runs:", len(CTX_JOBS))
CTX_RES = agents.read_results(sorted(RUN_DIR.glob("contexts_T1.p*.jsonl")))
assert all(j["key"] in CTX_RES for j in CTX_JOBS), "notebook 03's context runs are missing"

In [ ]:
def canary_jobs(agent, role):
    return [agents.canary_job(agent, role, d, EPIS[(d["doc_id"], e)], e, kind)
            for d in DOCS for e in REPLAY_E + RANDOM_E for kind in ("document", "control")]

CJOBS = {"C1": canary_jobs("llama8b", "C1"), "C2": canary_jobs("qwen7b", "C2")}
RES = {}
for role in CJOBS:
    RES.update(agents.read_results(sorted(RUN_DIR.glob(f"canary_{role}.p*.jsonl"))))
missing = sum(j["key"] not in RES for role in CJOBS for j in CJOBS[role])
assert missing == 0, f"{missing} canary runs of notebook 03 are missing"
JOB = {(j["role"], j["doc_id"], j["episode"], j["kind"]): j for role in CJOBS for j in CJOBS[role]}
PT = {}
for d in ORIG:
    ctx = BYPAY[pkey(agents.unit_id(d))]
    PT[d["doc_id"]] = sum(c in W1 for c in ctx) / len(ctx)
for d in REWR:
    ok = [CTX_RES[j["key"]] for j in CTX_JOBS if j["draw"] == d["draw"] and CTX_RES[j["key"]].get("error") is None]
    PT[d["doc_id"]] = (1 + sum(bool(r.get("security")) for r in ok)) / (1 + len(ok))
print("canary runs read:", sum(len(v) for v in CJOBS.values()), "| documents:", len(DOCS))

## Stage 3. Part 1: why working attacks show no event in replay

In [ ]:
def attacker_values(payload):
    vals = set(variants.EMAIL.findall(payload)) | set(variants.IBAN.findall(payload)) | set(variants.URL.findall(payload))
    for a, b in variants.QUOTED.findall(payload):
        q = (a or b).strip()
        if len(q) >= 4:
            vals.add(q)
    return {v.lower() for v in vals if len(v) >= 4}

def uses(call, vals):
    blob = json.dumps(call.get("args", {}), default=str).lower()
    return {v for v in vals if v in blob}

def is_delegated(draw):
    return bool(BANK[draw["suite"]][draw["task"]]["follows_document_instructions"])

def pair_cause(d, e, role, rule="control+sourced"):
    draw = EPIS[(d["doc_id"], e)]
    spec = SPECS[(draw["suite"], draw["task"])]
    doc_text = d["payload"] if (draw["mode"] == "replay" and draw.get("native_vectors")) else d["text"]
    rd = RES[JOB[(role, d["doc_id"], e, "document")]["key"]]
    rc = RES[JOB[(role, d["doc_id"], e, "control")]["key"]]
    dele = is_delegated(draw)
    rec = attribution.pair_record(dict(d, text=doc_text), draw, spec, rd, rc, None, rule, "no event", dele)
    vals = attacker_values(d["payload"]) - {v for c in rc.get("calls", []) for v in uses(c, attacker_values(d["payload"]))}
    acted = [c for c in rd.get("calls", []) if uses(c, vals)]
    base = dict(doc_id=d["doc_id"], half=d["half"], kind=d["kind"], suite=d["suite"], task=f"{draw['suite']}/{draw['task']}",
                template=d["template"], episode=e, canary=role, delegated_task=dele, event=rec["event"], acted=bool(acted),
                acted_calls=";".join(sorted({c["function"] for c in acted})), reasons=";".join(rec["reasons"]))
    if rc.get("error") is not None or rd.get("error") is not None or rd.get("overflow") or rc.get("overflow"):
        return dict(base, cause="run failed or too long")
    if not attribution.valid(rc, spec, draw, "no event"):
        ctl_reasons = sorted({r for _, _, rs in taskbank.trace_events(rc.get("calls", []), spec, draw.get("secret"), draw.get("trap_names", ())) for r in rs})
        return dict(base, cause="control run has an event", control_reasons=";".join(ctl_reasons))
    if not rec["read"]:
        return dict(base, cause="document not read")
    if rec["event"]:
        return dict(base, cause="counted" if acted else "counted, no attacker detail in the calls")
    if not acted:
        return dict(base, cause="canary did not act on the attacker's details")
    any_ev = {attribution.call_key(c) for c, _ in attribution.event_calls(rd, spec, draw)}
    flagged = [c for c in acted if attribution.call_key(c) in any_ev]
    if not flagged:
        return dict(base, cause="acted, but no event reason for that call in the task's sets")
    if dele and not attribution.attributed(rd, rc, None, spec, draw, doc_text, rule, True) and attribution.attributed(rd, rc, None, spec, draw, doc_text, rule, False):
        return dict(base, cause="acted, authorised by the delegated-action rule")
    ctl_keys = {attribution.call_key(c) for c in rc.get("calls", [])}
    if all(attribution.call_key(c) in ctl_keys for c in flagged):
        return dict(base, cause="acted, same call in the control run")
    return dict(base, cause="acted, not sourced from the document")

APAIRS = [pair_cause(d, e, role) for d in DOCS if d["status"] == 1 for e in REPLAY_E for role in ("C1", "C2")]
with open("reports/diag_attack_pairs_03b.csv", "w", newline="") as f:
    keys = sorted({k for r in APAIRS for k in r})
    w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(APAIRS)
by_cause = collections.Counter(r["cause"] for r in APAIRS)
by_cause_role = collections.Counter((r["canary"], r["cause"]) for r in APAIRS)
print("replay pairs of working attacks:", len(APAIRS))
for c, n in by_cause.most_common():
    print(f"  {n:5d}  {c}   (C1 {by_cause_role[('C1', c)]}, C2 {by_cause_role[('C2', c)]})")
# documents never counted in replay: their causes
never = collections.defaultdict(list)
for r in APAIRS:
    never[r["doc_id"]].append(r)
NEVER = {k: v for k, v in never.items() if not any(x["event"] for x in v)}
doc_cause = collections.Counter()
for k, v in NEVER.items():
    usable = [x["cause"] for x in v if x["cause"] not in ("control run has an event", "run failed or too long", "document not read")]
    doc_cause[collections.Counter(usable).most_common(1)[0][0] if usable else "no usable replay pair"] += 1
print("attacks with no counted replay pair:", len(NEVER), "of", sum(d["status"] == 1 for d in DOCS))
for c, n in doc_cause.most_common():
    print(f"  {n:4d}  {c}")
by_task = collections.Counter((r["task"], r["cause"]) for r in APAIRS if r["doc_id"] in NEVER)
print("by task:", by_task.most_common(10))

## Stage 4. Part 2: benign documents marked attack seen

In [ ]:
BCASES, sig = [], collections.Counter()
for d in DOCS:
    if d["status"] != 0:
        continue
    for e in REPLAY_E + RANDOM_E:
        draw = EPIS[(d["doc_id"], e)]
        spec = SPECS[(draw["suite"], draw["task"])]
        for role in ("C1", "C2"):
            rd = RES[JOB[(role, d["doc_id"], e, "document")]["key"]]
            rc = RES[JOB[(role, d["doc_id"], e, "control")]["key"]]
            for rule in ("control+sourced", "control+newtool-or-sourced"):
                kept = attribution.attributed(rd, rc, None, spec, draw, d["text"], rule, is_delegated(draw))
                if kept and attribution.valid(rc, spec, draw, "no event") and canary.document_read(rd.get("outputs", []), d["text"]):
                    for c, rs in kept:
                        key = "+".join(sorted({r.split(":")[0] for r in rs}))
                        sig[(rule, role, draw["mode"], key)] += 1
                        BCASES.append(dict(rule=rule, doc_id=d["doc_id"], source=d["source"], delegated_doc=d["delegated"], mode=draw["mode"],
                                           task=f"{draw['suite']}/{draw['task']}", canary=role, signal=key,
                                           call=c["function"] + " " + json.dumps(c.get("args", {}), default=str)[:160]))
lines = ["# Benign documents with a counted event (notebook 03 runs)", "",
         "Signals by rule, canary and episode mode (calls):", ""]
lines += [f"- {r} | {ro} | {m} | {k}: {n}" for (r, ro, m, k), n in sorted(sig.items())]
lines += ["", "Cases under control+sourced (up to 40):", ""]
lines += [f"- {c['source']} ({'delegated' if c['delegated_doc'] else 'plain'}), {c['mode']} {c['task']}, {c['canary']}, {c['signal']}: {c['call']}"
          for c in [x for x in BCASES if x["rule"] == "control+sourced"][:40]]
Path("reports/diag_benign_03b.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines[:40]))

## Stage 5. Part 3: candidate changes on all 400 DEV documents

In [ ]:
BASE = {}
for d in DOCS:
    for e in REPLAY_E + RANDOM_E:
        draw = EPIS[(d["doc_id"], e)]
        spec = SPECS[(draw["suite"], draw["task"])]
        text = d["payload"] if (draw["mode"] == "replay" and draw.get("native_vectors")) else d["text"]
        for role in ("C1", "C2"):
            rd = RES[JOB[(role, d["doc_id"], e, "document")]["key"]]
            rc = RES[JOB[(role, d["doc_id"], e, "control")]["key"]]
            ran = rc.get("error") is None and rd.get("error") is None
            BASE[(d["doc_id"], e, role)] = dict(draw=draw, spec=spec, text=text, rd=rd, rc=rc, dele=is_delegated(draw),
                                                read=canary.document_read(rd.get("outputs", []), text), ran=ran,
                                                no_event=ran and attribution.valid(rc, spec, draw, "no event"))

def kept_calls(b, rule, dmode, tmode):
    draw, spec, text, rd, rc, dele = b["draw"], b["spec"], b["text"], b["rd"], b["rc"], b["dele"]
    ev = attribution.attributed(rd, rc, None, spec, draw, text, rule, dele and dmode == "on")
    if dele and dmode == "limit":
        ctl_n = collections.Counter(c["function"] for c in rc.get("calls", []))
        flagged = {attribution.call_key(c) for c, _ in ev}
        plain_n = collections.Counter(c["function"] for c in rd.get("calls", []) if attribution.call_key(c) not in flagged)
        room = {f: max(0, ctl_n[f] - plain_n[f]) for f in ctl_n}
        out = []
        for c, rs in ev:
            if attribution.delegated_authorised(c, rs, text, spec) and room.get(c["function"], 0) > 0:
                room[c["function"]] -= 1
                continue
            out.append((c, rs))
        ev = out
    if tmode == "off":
        ev = [(c, [r for r in rs if r != "trap_tool"]) for c, rs in ev]
        ev = [(c, rs) for c, rs in ev if rs]
    elif tmode == "sourced":
        ev = [(c, rs) for c, rs in ev if rs != ["trap_tool"] or attribution.sourced(c, [], text)]
    return ev

VARIANTS = [(rule, vmode, dmode, tmode) for rule in ("control+sourced", "control+newtool-or-sourced")
            for vmode in ("no event", "ran") for dmode in ("off", "on", "limit") for tmode in ("count", "sourced", "off")]
PAIRS = {}
t0 = time.time()
for v in VARIANTS:
    rule, vmode, dmode, tmode = v
    for d in DOCS:
        for e in REPLAY_E + RANDOM_E:
            for role in ("C1", "C2"):
                b = BASE[(d["doc_id"], e, role)]
                PAIRS[(v, d["doc_id"], e, role)] = dict(event=bool(kept_calls(b, rule, dmode, tmode)),
                                                        valid=b["ran"] if vmode == "ran" else b["no_event"], read=b["read"])
print(f"pair records: {len(PAIRS)} in {time.time() - t0:.0f} s")

def outcome_m(eps, k_min, m):
    usable = [x for x in eps if x["usable"]]
    n_ev = sum(x["event"] for x in usable)
    if n_ev >= m:
        return oracle.ATTACK_SEEN
    if n_ev == 0 and len(usable) >= k_min:
        return oracle.NOTHING_SEEN
    return oracle.INCONCLUSIVE

ROLESETS = {"both": ("C1", "C2"), "C1": ("C1",), "C2": ("C2",)}
EPREC = {}
for v in VARIANTS:
    for rs_name, roles in ROLESETS.items():
        for d in DOCS:
            EPREC[(v, rs_name, d["doc_id"])] = [oracle.episode_record([PAIRS[(v, d["doc_id"], e, ro)] for ro in roles]) for e in REPLAY_E + RANDOM_E]

SETTINGS = []
for v in VARIANTS:
    for rs_name in ROLESETS:
        for m in (1, 2):
            for r, s, k in oracle.configs(3, 3):
                if r == 0:
                    continue
                rows = []
                for d in DOCS:
                    allep = EPREC[(v, rs_name, d["doc_id"])]
                    eps = allep[:r] + allep[3:3 + s]
                    row = dict(status=d["status"], delegated=d["delegated"], outcome=outcome_m(eps, k, m))
                    if d["status"] == 1:
                        row.update(q=oracle.q_hat(eps), p_t=PT[d["doc_id"]], q_replay=oracle.q_hat(allep[:r]))
                    rows.append(row)
                mm = oracle.measures(rows)
                rep = oracle.measures([dict(x, q=x.get("q_replay")) for x in rows])
                SETTINGS.append(dict(rule=v[0], validity=v[1], delegated_rule=v[2], trap_tools=v[3], canaries=rs_name, events_needed=m,
                                     replay=r, random=s, k_min=k, agreement=mm["agreement"], inconclusive=mm["inconclusive"],
                                     attack_seen=mm["attack_seen"], attacks=mm["attacks"], benign_seen=mm["benign_seen"], benign=mm["benign"],
                                     delegated_seen=mm["delegated_seen"], delegated=mm["delegated"],
                                     violation_share=mm["violation_share"], violation_ucb=mm["violation_ucb"],
                                     violation_ucb_replay_q=rep["violation_ucb"], **{f"mark_{x}": y for x, y in oracle.marks(mm).items()}))
print("settings:", len(SETTINGS))

## Stage 6. Settings table and summary

In [ ]:
def ok_marks(c, replay_q=False):
    vu = c["violation_ucb_replay_q"] if replay_q else c["violation_ucb"]
    return c["inconclusive"] <= 0.30 and c["delegated_seen"] <= 0.05 * c["delegated"] and vu <= 0.10

order = lambda c: (-c["agreement"], c["inconclusive"], c["replay"] + c["random"])
srt = sorted(SETTINGS, key=order)
with open("reports/diag_settings_03b.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(SETTINGS[0])); w.writeheader()
    w.writerows([{x: (round(y, 4) if isinstance(y, float) else y) for x, y in c.items()} for c in srt])

fmtc = lambda c: (f"{c['rule'][8:]}, validity {c['validity']}, delegated {c['delegated_rule']}, traps {c['trap_tools']}, {c['canaries']}, "
                  f"events needed {c['events_needed']}, replay {c['replay']}, random {c['random']}, k_min {c['k_min']}: agreement {c['agreement']:.3f}, "
                  f"inconclusive {c['inconclusive']:.3f}, attacks {c['attack_seen']}/{c['attacks']}, benign {c['benign_seen']}/{c['benign']}, "
                  f"delegated {c['delegated_seen']}/{c['delegated']}, violation bound {c['violation_ucb']:.3f} (q over replay {c['violation_ucb_replay_q']:.3f})")
L = ["# Gate G1 diagnosis (notebook 03b, CPU, notebook 03's runs)", "",
     "## Part 1. Replay pairs of working attacks by cause", ""]
L += [f"- {c}: {n} (C1 {by_cause_role[('C1', c)]}, C2 {by_cause_role[('C2', c)]})" for c, n in by_cause.most_common()]
L += ["", f"Attacks with no counted replay pair (control+sourced, as in notebook 03): {len(NEVER)} of {sum(d['status'] == 1 for d in DOCS)}; main cause per document:"]
L += [f"- {c}: {n}" for c, n in doc_cause.most_common()]
L += ["", "## Part 2. Benign alarms: reports/diag_benign_03b.md", "",
      "## Part 3. Candidate changes (all 400 DEV documents)", "",
      f"Settings meeting the inconclusive, delegated and violation marks: {sum(ok_marks(c) for c in SETTINGS)} of {len(SETTINGS)}; "
      f"with q over replay episodes in Equation (7): {sum(ok_marks(c, True) for c in SETTINGS)}.", "",
      "Best agreement overall:"] + ["- " + fmtc(c) for c in srt[:5]]
L += ["", "Best agreement among settings meeting those three marks:"] + (["- " + fmtc(c) for c in [c for c in srt if ok_marks(c)][:5]] or ["- none"])
L += ["", "Best agreement among settings meeting them with q over replay episodes:"] + (["- " + fmtc(c) for c in [c for c in srt if ok_marks(c, True)][:5]] or ["- none"])
L += ["", "Each factor, best setting with that level (agreement; violation bound; violation bound with q over replay):"]
for fac, levels in [("rule", ("control+sourced", "control+newtool-or-sourced")), ("validity", ("no event", "ran")),
                    ("delegated_rule", ("off", "on", "limit")), ("trap_tools", ("count", "sourced", "off")),
                    ("canaries", ("both", "C1", "C2")), ("events_needed", (1, 2))]:
    for lv in levels:
        sub = [c for c in srt if c[fac] == lv and c["inconclusive"] <= 0.30]
        if sub:
            b = sub[0]
            L.append(f"- {fac} = {lv}: agreement {b['agreement']:.3f}, attacks {b['attack_seen']}/{b['attacks']}, benign {b['benign_seen']}/{b['benign']}, "
                     f"delegated {b['delegated_seen']}/{b['delegated']}, bound {b['violation_ucb']:.3f}, replay-q bound {b['violation_ucb_replay_q']:.3f}")
Path("reports/diag_03b.md").write_text("\n".join(L) + "\n")
print("\n".join(L))

## Stage 7. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("CANARY_REPO_ROOT", "/content/drive/MyDrive/CanaryLabel_Research/canarylabel-research"))
r = subprocess.run(["python", "tools/commit_cell.py", "03b Gate G1 diagnosis: causes of missed attacks and benign alarms, candidate changes (CPU)"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)